# 14 — Week 4: Chronos-Bolt (small) zero-shot on the full validation split

Same usable origins and metrics as all other models (`ok_h`, `split_h`, `src/evaluation/metrics.py`).
Point forecast = median; the 10% and 90% quantiles are stored for the uncertainty work (RQ3).
Context length is a hyperparameter chosen on validation: 60 (protocol window), 512 and 2048 minutes (Chronos-Bolt's
maximum); invalid minutes inside a context are passed as NaN. The model is deterministic (one run, no seeds).
Secondary result (leakage caveat). Outputs: `experiments/week4_fm/`, `reports/week4/fm_val_summary.csv`,
`reports/week4/benchmark_val_week4_all.csv`. **No test data is touched.** Runtime: about 10 minutes on MPS.

In [1]:
# Cell 1: setup
import os
os.environ["HF_HUB_OFFLINE"] = "1"
import sys, json, time, warnings
from pathlib import Path
import numpy as np
import polars as pl
import torch

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
from models import rnn as R
from evaluation import metrics as M
from tracking import log_run
from chronos import BaseChronosPipeline

MODEL_ID, NAME = "amazon/chronos-bolt-small", "chronos_bolt_small"
CONTEXTS = [60, 512, 2048]
QL = [0.1, 0.5, 0.9]
H_MAX = max(R.HORIZONS)
DEV = "mps" if torch.backends.mps.is_available() else "cpu"
GROUP = "week4_fm"
REP = ROOT / "reports" / "week4"
pipe = BaseChronosPipeline.from_pretrained(MODEL_ID, device_map=DEV, torch_dtype=torch.float32)
n_params = int(sum(p.numel() for p in pipe.model.parameters()))
md = pl.read_parquet(ROOT / "data" / "processed" / "model_dataset.parquet")
meta = json.loads((ROOT / "data" / "processed" / "model_dataset_meta.json").read_text())
data = R.Data(md, {"scaling": "local", "calendar": False, "covariates": False})
valid = md["valid"].to_numpy()
Pnan = np.where(valid, md["P_kW"].fill_null(np.nan).to_numpy(), np.nan).astype(np.float32)
L24 = md["L24_kW"].to_numpy().astype(float)
mod5 = md["mod5"].to_numpy().astype(int)
ep = md["minute_epoch"].to_numpy()
va = data.origins("val")
print(f"device {DEV} | params {n_params:,} | val origins {len(va):,}")


def forecast(ctx_len, t, batch=256):
    qs = []
    off = np.arange(-(ctx_len - 1), 1)
    for i in range(0, len(t), batch):
        idx = t[i:i + batch, None] + off[None, :]
        c = np.full(idx.shape, np.nan, dtype=np.float32)
        ok = idx >= 0
        c[ok] = Pnan[idx[ok]]
        q, _ = pipe.predict_quantiles(torch.from_numpy(c), prediction_length=H_MAX, quantile_levels=QL)
        qs.append(q.float().cpu().numpy())
    return np.concatenate(qs)                                       # (N, 15, 3)


Loading weights: 100%|████████████████████| 143/143 [00:00<00:00, 29779.33it/s]


device mps | params 47,718,016 | val origins 42,575


In [2]:
# Cell 2: full validation run per context length (resumable)
results = {}
for L in CONTEXTS:
    run = f"{NAME}_ctx{L}"
    out = ROOT / "experiments" / GROUP / run
    if (out / "metrics.json").exists():
        results[L] = json.loads((out / "metrics.json").read_text())
        print(f"{run}: done earlier, skipped")
        continue
    out.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    Q = forecast(L, va)
    infer_s = time.time() - t0
    rows, metrics, parts = [], {}, []
    for j, h in enumerate(R.HORIZONS):
        t = data.idx[h]["val"]
        pos = np.searchsorted(va, t)
        q = Q[pos, h - 1, :]
        yhat = q[:, 1].astype(float)
        y, p0 = data.P[t + h], data.P[t]
        ramp = md[f"ramp_h{h}"].to_numpy().astype(float)[t]
        r = M.evaluate(y, yhat, p0, l24=L24[t], ramp=ramp, thr=meta["thresholds"][str(h)]["rel_p95"],
                       mod5_target=mod5[t + h] if h == 1 else None, dm_lag=60)
        r["cover_q10_q90"] = float(np.mean((y >= q[:, 0]) & (y <= q[:, 2])))
        r["width_q10_q90"] = float(np.mean(q[:, 2] - q[:, 0]))
        rows.append({"h": h, **r})
        metrics.update({f"h{h}/{k}": v for k, v in r.items() if isinstance(v, (int, float))})
        parts.append(pl.DataFrame({"minute_epoch": ep[t], "h": np.full(len(t), h, np.int16), "y": y, "p0": p0,
                                   "yhat": yhat, "q10": q[:, 0].astype(float), "q90": q[:, 2].astype(float)}))
    pl.concat(parts).write_parquet(out / "predictions_val.parquet")
    info = {"model": MODEL_ID, "context": L, "n_params": n_params, "train_s": 0.0, "device": DEV,
            "infer_s_val": infer_s, "infer_ms_per_sample": 1000 * infer_s / len(va)}
    metrics.update({k: v for k, v in info.items() if isinstance(v, (int, float))})
    res = {"info": info, "val": rows, "metrics": metrics}
    log_run(ROOT, run, GROUP, {"family": "foundation_zero_shot", **info}, metrics, tables={"val_metrics": pl.DataFrame(rows)})
    (out / "metrics.json").write_text(json.dumps(res, indent=1, default=str))
    results[L] = res
    print(f"{run}: {infer_s / 60:.1f} min | val MAE " + " / ".join(f"{r_['MAE']:.4f}" for r_ in rows) +
          " | skill " + " / ".join(f"{r_['skill_MAE']:+.3f}" for r_ in rows) +
          " | 80% interval coverage " + " / ".join(f"{r_['cover_q10_q90']:.3f}" for r_ in rows))


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/aryaneuse/.netrc.


chronos_bolt_small_ctx60: 0.4 min | val MAE 0.2770 / 0.4352 / 0.6509 | skill +0.033 / -0.012 / -0.076 | 80% interval coverage 0.809 / 0.803 / 0.743
chronos_bolt_small_ctx512: 1.2 min | val MAE 0.2808 / 0.4163 / 0.5955 | skill +0.020 / +0.032 / +0.015 | 80% interval coverage 0.811 / 0.818 / 0.802
chronos_bolt_small_ctx2048: 5.6 min | val MAE 0.2974 / 0.4291 / 0.5988 | skill -0.038 / +0.002 / +0.010 | 80% interval coverage 0.836 / 0.826 / 0.809


In [3]:
# Cell 3: choose the context on validation (mean MAE over horizons) and add Chronos to the benchmark table
summ = pl.DataFrame([{"model": f"{NAME}_ctx{L}", "context": L, "h": r_["h"], "MAE": r_["MAE"], "RMSE": r_["RMSE"],
                      "sMAPE": r_["sMAPE"], "R2_delta": r_["R2_delta"], "skill_MAE": r_["skill_MAE"],
                      "MAE_ramp": r_["MAE_ramp"], "MAE_normal": r_["MAE_normal"], "ramp_F1": r_["ramp_F1"],
                      "DM_p": r_["DM_p"], "cover_q10_q90": r_["cover_q10_q90"], "width_q10_q90": r_["width_q10_q90"],
                      "infer_ms_per_sample": res["info"]["infer_ms_per_sample"]}
                     for L, res in results.items() for r_ in res["val"]])
summ.write_csv(REP / "fm_val_summary.csv")
pl.Config.set_tbl_rows(30); pl.Config.set_tbl_cols(16); pl.Config.set_tbl_width_chars(220)
print(summ.select("h", "context", pl.col("MAE").round(4), pl.col("skill_MAE").round(3), pl.col("MAE_ramp").round(3),
                  pl.col("MAE_normal").round(4), pl.col("ramp_F1").round(3), pl.col("cover_q10_q90").round(3),
                  pl.col("width_q10_q90").round(3), pl.col("infer_ms_per_sample").round(3)).sort("h", "context"))
best_L = summ.group_by("context").agg(pl.col("MAE").mean()).sort("MAE")["context"][0]
print(f"\ncontext chosen on validation (mean MAE over h): {best_L} min")
bench = pl.read_csv(REP / "benchmark_val_week4_nf.csv")
fm = (summ.filter(pl.col("context") == best_L)
          .with_columns(pl.lit("foundation").alias("family"), pl.lit(NAME).alias("model"), pl.lit(False).alias("covariates"),
                        pl.lit(1).alias("n_seeds"), pl.col("MAE").alias("MAE_mean"), pl.lit(0.0).alias("MAE_sd"))
          .select(bench.columns))
allb = pl.concat([bench, fm.with_columns([pl.col(c).cast(bench.schema[c]) for c in bench.columns])]).sort("h", "MAE_mean")
allb.write_csv(REP / "benchmark_val_week4_all.csv")
for h in R.HORIZONS:
    t = allb.filter(pl.col("h") == h)
    rank = t["model"].to_list().index(NAME) + 1
    print(f"h={h:>2}: {NAME} rank {rank} of {t.height} | MAE {t.filter(pl.col('model') == NAME)['MAE_mean'][0]:.4f} "
          f"| best {t['model'][0]} {t['MAE_mean'][0]:.4f}")
print("written:", REP / "fm_val_summary.csv", "|", REP / "benchmark_val_week4_all.csv")


shape: (9, 10)
┌─────┬─────────┬────────┬───────────┬──────────┬────────────┬─────────┬───────────────┬───────────────┬─────────────────────┐
│ h   ┆ context ┆ MAE    ┆ skill_MAE ┆ MAE_ramp ┆ MAE_normal ┆ ramp_F1 ┆ cover_q10_q90 ┆ width_q10_q90 ┆ infer_ms_per_sample │
│ --- ┆ ---     ┆ ---    ┆ ---       ┆ ---      ┆ ---        ┆ ---     ┆ ---           ┆ ---           ┆ ---                 │
│ i64 ┆ i64     ┆ f64    ┆ f64       ┆ f64      ┆ f64        ┆ f64     ┆ f64           ┆ f64           ┆ f64                 │
╞═════╪═════════╪════════╪═══════════╪══════════╪════════════╪═════════╪═══════════════╪═══════════════╪═════════════════════╡
│ 1   ┆ 60      ┆ 0.277  ┆ 0.033     ┆ 1.421    ┆ 0.2248     ┆ 0.165   ┆ 0.809         ┆ 0.883         ┆ 0.512               │
│ 1   ┆ 512     ┆ 0.2808 ┆ 0.02      ┆ 1.44     ┆ 0.2281     ┆ 0.137   ┆ 0.811         ┆ 0.868         ┆ 1.622               │
│ 1   ┆ 2048    ┆ 0.2974 ┆ -0.038    ┆ 1.467    ┆ 0.2444     ┆ 0.104   ┆ 0.836         ┆ 0.987  

In [5]:
# Cell 4: export outputs (save the notebook first with Cmd+S)
%run ../src/tools/nb_outputs_to_md.py 14_week4_chronos_val.ipynb


written 14_week4_chronos_val_outputs.md | code cells 4 | figures 0
